Validação dos fluxos turísticos

Brasil ↔ Coreia do Sul · 1998–2025

--------------------

Se todos os anos possuem 12 meses para cada fluxo.

Se a soma das chegadas mensais corresponde ao total anual.

Se existem divergências ou registros incompletos.

Se os indicadores estão prontos para as análises posteriores.

In [0]:
from pyspark.sql import functions as F

df_anual = spark.table("chegadas.gold.turismo_bilateral_anual")
df_mensal = spark.table("chegadas.gold.turismo_bilateral_mensal")

In [0]:
df_soma_mensal = (
    df_mensal

    .groupBy("ano", "origem", "destino")

    .agg(
        F.sum("chegadas_observadas").alias(
            "soma_meses_observados"
        ),

        F.countDistinct("mes").alias(
            "meses_disponiveis"
        ),

        F.sum("registros_nulos").alias(
            "registros_nulos_mensais"
        )
    )
)

display(
    df_soma_mensal
    .orderBy("ano", "origem")
)

In [0]:
df_validacao = (
    df_anual.alias("a")

    .join(
        df_soma_mensal.alias("m"),
        on=["ano", "origem", "destino"],
        how="full"
    )

    .select(
        "ano",
        "origem",
        "destino",

        F.col("a.chegadas_observadas").alias(
            "total_anual"
        ),

        "soma_meses_observados",
        "meses_disponiveis",
        "registros_nulos_mensais"
    )

    .withColumn(
        "diferenca",
        F.col("total_anual") -
        F.col("soma_meses_observados")
    )

    .orderBy("ano", "origem")
)

display(df_validacao)

In [0]:
df_validacao = (
    df_validacao

    .withColumn(
        "status_validacao",

        F.when(
            F.col("total_anual").isNull() |
            F.col("soma_meses_observados").isNull(),
            "dados_ausentes"
        )

        .when(
            F.col("meses_disponiveis") != 12,
            "meses_incompletos"
        )

        .when(
            F.col("diferenca") != 0,
            "divergente"
        )

        .when(
            F.col("registros_nulos_mensais") > 0,
            "coincidente_com_nulos"
        )

        .otherwise("coincidente")
    )
)

display(
    df_validacao
    .groupBy("status_validacao")
    .count()
    .orderBy("status_validacao")
)

In [0]:
display(
    df_validacao
    .filter(
        F.col("status_validacao") != "coincidente"
    )
    .orderBy("ano", "origem")
)

In [0]:
df_validacao_final = (
    df_validacao
    .filter(F.col("ano").between(1998, 2025))
)

display(
    df_validacao_final
    .groupBy("status_validacao")
    .count()
    .orderBy("status_validacao")
)

In [0]:
assert df_validacao_final.count() == 56, (
    "Quantidade inesperada de comparações."
)

assert (
    df_validacao_final
    .filter(
        F.col("status_validacao") != "coincidente"
    )
    .filter(
        F.col("status_validacao") != "coincidente_com_nulos"
    )
    .count()
) == 0, "Existem divergências ou dados ausentes."

print("Validação cruzada concluída!")


## Resultado da validação cruzada

Foi realizada a comparação entre os totais anuais e a soma
das chegadas mensais dos dois fluxos turísticos.

Período analisado: 1998–2025.

Foram verificadas 56 combinações de ano e fluxo turístico.

Resultados:
- 53 comparações apresentaram coincidência numérica sem
  valores nulos nos registros mensais.
- 3 comparações apresentaram coincidência numérica, mas
  continham registros com valores nulos.
- Nenhuma divergência numérica foi identificada.

Os registros com valores nulos pertencem ao fluxo
Coreia do Sul → Brasil, nos anos de 1999, 2012 e 2014.

Esses totais representam a soma dos valores disponíveis
e não devem ser interpretados como necessariamente completos.

O ano de 1996 também apresenta registros nulos na base
brasileira, mas está fora do período de comparação mensal.

Os anos de 1990 a 1997 foram excluídos somente da validação
cruzada, pois não fazem parte do período mensal comum.

As tabelas Gold anual e mensal permanecem preservadas.

Documnetação

In [0]:
spark.sql("""
COMMENT ON TABLE chegadas.gold.turismo_bilateral_anual IS
'Gold - Fluxo turistico bilateral anual entre Brasil e
Coreia do Sul, de 1990 a 2025.

Contem um registro por ano e direcao do fluxo turistico.

Os totais brasileiros foram calculados a partir dos
registros de chegadas por mes, UF e via de acesso.

Os totais coreanos foram obtidos dos registros anuais
da fonte original.

Nos anos com registros nulos, chegadas permanece NULL
e chegadas_observadas representa a soma dos valores
disponiveis.

Os dois fluxos provem de fontes estatisticas distintas
e sua comparabilidade metodologica deve ser considerada.'
""")

In [0]:
spark.sql("""
COMMENT ON TABLE chegadas.gold.turismo_bilateral_mensal IS
'Gold - Fluxo turistico bilateral mensal entre Brasil
e Coreia do Sul, de 1998 a 2025.

Contem um registro por ano, mes e direcao do fluxo.

Os meses foram padronizados numericamente de 1 a 12.

As chegadas de turistas coreanos ao Brasil foram
agregadas por ano e mes.

Os registros de brasileiros na Coreia foram selecionados
exclusivamente da granularidade mensal.

Os valores ausentes foram preservados e identificados
por meio das colunas de auditoria.

A soma mensal foi comparada aos totais anuais na
validacao cruzada da Gold.'
""")

In [0]:
COMENTARIOS_GOLD = {
    "ano": "Ano de referencia das chegadas internacionais.",

    "origem": "Pais de origem do fluxo turistico analisado.",

    "destino": "Pais de destino do fluxo turistico analisado.",

    "chegadas": "Total de chegadas. Permanece NULL quando existem registros de origem com valores ausentes.",

    "chegadas_observadas": "Soma das chegadas disponiveis, incluindo periodos com registros parcialmente preenchidos.",

    "total_registros": "Quantidade de registros da fonte utilizados na construcao do indicador.",

    "registros_nulos": "Quantidade de registros da fonte com valores ausentes na coluna de chegadas.",

    "status_dados": "Indica se existem registros nulos entre os dados utilizados no indicador."
}

TABELAS_GOLD = [
    "chegadas.gold.turismo_bilateral_anual",
    "chegadas.gold.turismo_bilateral_mensal"
]

for tabela in TABELAS_GOLD:

    for coluna, comentario in COMENTARIOS_GOLD.items():

        spark.sql(
            f"ALTER TABLE {tabela} "
            f"ALTER COLUMN `{coluna}` COMMENT '{comentario}'"
        )

# Documentar a coluna exclusiva da tabela mensal

spark.sql("""
ALTER TABLE chegadas.gold.turismo_bilateral_mensal
ALTER COLUMN mes COMMENT
'Mes de referencia das chegadas internacionais,
representado por um numero inteiro de 1 a 12.'
""")

print("Documentação das tabelas Gold concluída!")

In [0]:
df_anual = spark.table(
    "chegadas.gold.turismo_bilateral_anual"
)

df_mensal = spark.table(
    "chegadas.gold.turismo_bilateral_mensal"
)

print("AUDITORIA GOLD")

print("Registros anuais:", df_anual.count())
print("Registros mensais:", df_mensal.count())

assert df_anual.count() == 72
assert df_mensal.count() == 672

# Conferir unicidade dos registros anuais

duplicados_anuais = (
    df_anual
    .groupBy("ano", "origem", "destino")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Conferir unicidade dos registros mensais

duplicados_mensais = (
    df_mensal
    .groupBy("ano", "mes", "origem", "destino")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicados_anuais == 0
assert duplicados_mensais == 0

print("Unicidade dos registros: OK")
print("Auditoria final concluída!")


## Conclusão da camada Gold

Foram construídas duas tabelas analíticas para o estudo
do turismo bilateral entre Brasil e Coreia do Sul.

### Tabela anual

Período: 1990–2025.

Total de registros: 72.

Granularidade: um registro por ano e direção do fluxo.

### Tabela mensal

Período: 1998–2025.

Total de registros: 672.

Granularidade: um registro por ano, mês e direção
do fluxo turístico.

### Validação cruzada

Foram realizadas 56 comparações entre os totais anuais
e a soma dos respectivos meses, no período de 1998 a 2025.

Resultados:
- 53 comparações coincidentes sem valores nulos.
- 3 comparações coincidentes com registros nulos.
- Nenhuma divergência numérica identificada.

Os registros incompletos foram identificados e preservados.

A coincidência entre os totais anuais e mensais não
garante a completude estatística das fontes originais.

### Status

Camadas Bronze, Silver e Gold concluídas.

Próxima etapa: análise exploratória dos dados com SQL
e Python.